In [ ]:
# import lib

In [1]:
import requests
import io
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Task 1

In [2]:
url = "https://raw.githubusercontent.com/madmanor/lis4893/refs/heads/main/lab-4/SMSSpamCollection"
response = requests.get(url)
response.raise_for_status()
text = response.text

In [ ]:
# load into dataframe

In [10]:
df = pd.read_csv(io.StringIO(text), sep='\t', names=['label', 'message'])
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [4]:
# map the labels to ints, ham=0 and spam=1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})


In [20]:
features = df['message']
target = df['label']

In [19]:
target_names = ['Ham', 'Spam']

In [6]:
# Task 2 Split data 70;30

In [7]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.3, random_state=42)

# set empty valyes to strings
X_train = X_train.fillna('')
X_test = X_test.fillna('')

In [8]:
preprocessing = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer())
])

In [9]:
train_preprocessed = preprocessing.fit_transform(X_train)
test_preprocessed = preprocessing.transform(X_test)

In [ ]:
# Task 3 Classifiers

In [11]:
nb_classifier = MultinomialNB()
svm_classifier = LinearSVC()
lr_classifier = LogisticRegression(multi_class="ovr")
ada_classifier = AdaBoostClassifier()
random_classifier = RandomForestClassifier()

In [ ]:
# training models

In [12]:
nb_classifier.fit(train_preprocessed, y_train)
svm_classifier.fit(train_preprocessed, y_train)
lr_classifier.fit(train_preprocessed, y_train)
ada_classifier.fit(train_preprocessed, y_train)
random_classifier.fit(train_preprocessed, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


RandomForestClassifier()

In [ ]:
# predictions

In [22]:
nb_predictions = nb_classifier.predict(test_preprocessed)
lr_predictions = lr_classifier.predict(test_preprocessed)
ada_predictions = ada_classifier.predict(test_preprocessed)
rf_predictions = random_classifier.predict(test_preprocessed)

In [ ]:
# reports

In [26]:
print("naive bayes")
print(classification_report(y_test, nb_predictions, target_names=target_names))
print("logistic regression")
print(classification_report(y_test, lr_predictions, target_names=target_names))
print("ada")
print(classification_report(y_test, ada_predictions, target_names=target_names))


naive bayes
              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98      1448
        Spam       1.00      0.72      0.84       224

    accuracy                           0.96      1672
   macro avg       0.98      0.86      0.91      1672
weighted avg       0.96      0.96      0.96      1672

logistic regression
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98      1448
        Spam       0.99      0.80      0.88       224

    accuracy                           0.97      1672
   macro avg       0.98      0.90      0.93      1672
weighted avg       0.97      0.97      0.97      1672

ada
              precision    recall  f1-score   support

         Ham       0.96      0.99      0.98      1448
        Spam       0.94      0.72      0.82       224

    accuracy                           0.96      1672
   macro avg       0.95      0.86      0.90      1672
weighted avg       0.96      0.96     

In [ ]:
# logistic regression perfromed the best with the highest accuracy at 97% though all of them had very high accuracy
# it also had a recall of 80% on spam and 100% on ham, the recall on spam shows its better than the other 2.